# FedKG-Maint — Data Preparation (revised v3)

**Project:** Privacy-preserving predictive maintenance with federated learning, a maintenance knowledge graph and LLM explanations. This notebook prepares the Microsoft Azure Predictive Maintenance dataset only.

**This revised notebook writes to `prep_v3/`; it does not overwrite `prep_v2/`.** Existing outputs are intentionally kept as an audit trail. Outputs from the previous notebook have been cleared because its counts and test definitions are no longer current.

## Key protocol decisions

| Topic | Decision |
|---|---|
| Target | `label_failure_24h = 1` iff a recorded failure occurs in `(t, t + 24h]`. Drop the last 24 hours per machine because the horizon is not observable. |
| Simulated factories | Five deterministic, non-IID clients with 20 machines each. Every primary client contains multiple machine models; model proportions deliberately differ across clients. |
| Alternate shift scenario | Save a separate model-blocked manifest for a stronger machine-type-shift stress test. It is not the primary split and must be evaluated separately. |
| Held-out factory | `factory_5` contributes no supervised training, validation or tuning rows. |
| Local held-out adaptation | The first 50% of the timeline is used as a feature-only, unlabeled calibration window for `local_factory_5`. Its labels are not written to the calibration file. |
| Evaluation sets | `test_site_full` includes all held-out-factory rows **after** calibration; `test_site_late` is the late-window subset and is the strictest site-plus-time test; `test_time_late` uses development factories 1–4 in the late window. |
| Temporal splits | Development factories 1–4: train / validation / late test, with 24 h purge gaps. |
| Features | 25 past-only features in five groups: raw sensors, rolling mean/std (24 h, 7 d), error recency, maintenance recency, history counts. |
| Normalization | `global` fits one scaler on pooled training rows. `local` fits per-client scalers; the held-out client's scaler uses only its earlier feature-only calibration file. Calibration rows are disjoint from every evaluation split. |
| Integrity | Raw SHA-256 hashes are compared with a prior manifest when available; a changed dataset fails closed rather than silently replacing the reference. |

## Outputs (`prep_v3/`)

`processed/` (parquet splits and feature-only calibration data), `scalers/`, `configs/`, `manifests/`, `reports/`, and `fedkg_data.py`.

**Important:** This is still a centralized data-preparation simulation. Local feature/scaler policies do not by themselves establish end-to-end privacy; later federated-learning code must enforce the client boundary.


## 1. Setup

Environment, configuration, directories under `prep_v3/`, seeds, and the fail-fast check registry. The previous `prep_v2/` outputs are left untouched so old results remain auditable.


In [ ]:
# 1.1  Setup: environment, configuration, directories, seeds, check registry
import os, sys, json, random, hashlib, shutil, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

try:  # Google Colab
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:  # local run / testing
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

# ---- single project configuration ---------------------------------------------------------
SEED = 42
HORIZON_HOURS = 24
ROLLING_WINDOWS = {"24h": 24, "7d": 168}           # window name -> number of hourly rows
SENSORS = ["volt", "rotate", "pressure", "vibration"]
TARGET = "label_failure_24h"
N_FACTORIES, MACHINES_PER_FACTORY = 5, 20
HELD_OUT = "factory_5"
TRAINING_FACTORIES = [f"factory_{i}" for i in range(1, N_FACTORIES + 1) if f"factory_{i}" != HELD_OUT]
TRAIN_FRACTION = 0.80                              # share of the timeline used for training
VAL_FRACTION_OF_REMAINDER = 0.50                   # remainder is split validation / late test
FRAGILE_EVENT_THRESHOLD = 30                       # fewer distinct failure events => unstable metrics
HELD_OUT_CALIBRATION_FRACTION = 0.50                # earlier, unlabeled calibration window for local_factory_5

H = pd.Timedelta(hours=HORIZON_HOURS)
H64 = H.to_timedelta64()

# ---- write to a new versioned directory; old prep_v2 outputs remain untouched -------
RAW_DIR = PROJECT_DIR / "data" / "raw"
PREP_DIR = PROJECT_DIR / "prep_v3"
PROC_DIR, SCALER_DIR = PREP_DIR / "processed", PREP_DIR / "scalers"
CFG_DIR, REP_DIR, MAN_DIR = PREP_DIR / "configs", PREP_DIR / "reports", PREP_DIR / "manifests"
for d in [RAW_DIR, PROC_DIR, SCALER_DIR, CFG_DIR, REP_DIR, MAN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)

# ---- check registry: every check is recorded and stops the run on failure -----------------
CHECKS = {}
def record(check_id, ok, detail=""):
    CHECKS[check_id] = {"status": "PASS" if ok else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {check_id}: {detail}")
    if not ok:
        raise AssertionError(f"Check failed: {check_id} — {detail}")

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
print("Project directory:", PROJECT_DIR)
print("Output directory :", PREP_DIR)

## 2. Data acquisition and immutable fingerprints

The five Azure PdM CSVs are saved to `FedKG-Maint/data/raw/`. If they are missing or have unexpected row counts, the dataset is downloaded. Current SHA-256 hashes are checked against an existing manifest (`prep_v3/` first, then an earlier `prep_v2/` manifest if available). Existing fingerprints are never silently overwritten.


In [ ]:
# 2.1  Acquire the five raw CSVs and verify their fingerprints
EXPECTED_ROWS = {
    "PdM_telemetry.csv": 876_100, "PdM_errors.csv": 3_919, "PdM_failures.csv": 761,
    "PdM_maint.csv": 3_286, "PdM_machines.csv": 100,
}


def raw_is_ready():
    for name, n in EXPECTED_ROWS.items():
        p = RAW_DIR / name
        if not p.exists():
            return False
        try:
            if len(pd.read_csv(p, usecols=[0])) != n:
                return False
        except Exception:
            return False
    return True


if not raw_is_ready():
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    src_root = Path(kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance"))
    for name in EXPECTED_ROWS:
        found = list(src_root.rglob(name))
        assert len(found) == 1, f"{name}: expected exactly one match, found {len(found)}"
        shutil.copy2(found[0], RAW_DIR / name)
    print("Downloaded raw data to", RAW_DIR)
else:
    print("Raw data already present:", RAW_DIR)


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


checksums = {
    name: {"rows": EXPECTED_ROWS[name], "bytes": (RAW_DIR / name).stat().st_size,
           "sha256": sha256_of(RAW_DIR / name)}
    for name in EXPECTED_ROWS
}
record("DATA-1 raw files present with expected row counts", raw_is_ready(),
       ", ".join(f"{n.replace('PdM_', '').replace('.csv', '')}={v['rows']:,}" for n, v in checksums.items()))

# Prefer this version's immutable reference. If this is the first prep_v3 run, compare
# against a previous prep_v2 manifest when available. Do not overwrite a mismatching reference.
checksum_path = MAN_DIR / "raw_data_checksums.json"
reference_candidates = [
    checksum_path,
    PROJECT_DIR / "prep_v2" / "manifests" / "raw_data_checksums.json",
    PROJECT_DIR / "manifests" / "raw_data_checksums.json",
]
reference_path = next((p for p in reference_candidates if p.exists()), None)
if reference_path is not None:
    reference = json.loads(reference_path.read_text())
    mismatches = []
    for name, actual in checksums.items():
        expected = reference.get(name)
        if expected is None:
            mismatches.append(f"{name}: missing from reference manifest")
            continue
        for field in ("rows", "bytes", "sha256"):
            if expected.get(field) != actual[field]:
                mismatches.append(f"{name}: {field} differs")
    record("DATA-2 raw SHA-256 fingerprints match the saved reference",
           not mismatches,
           f"reference={reference_path}; " + ("all fingerprints match" if not mismatches else "; ".join(mismatches)))
    # When using a previous-version reference, copy it only after successful comparison.
    if reference_path != checksum_path:
        checksum_path.write_text(json.dumps(reference, indent=2))
else:
    # First-ever run: initialize a baseline. Future runs will compare against it.
    checksum_path.write_text(json.dumps(checksums, indent=2))
    record("DATA-2 initial raw fingerprint baseline created", True,
           "No earlier checksum manifest existed; subsequent runs will fail if any hash changes.")

print("Fingerprint manifest:", checksum_path)


## 3. Data-quality audit
Missing values, duplicates, regular hourly telemetry grid, alignment of failure events with the telemetry grid, and identifier consistency across the five tables.

In [ ]:
# 3.1  Load raw tables and run the data-quality audit
tele = pd.read_csv(RAW_DIR / "PdM_telemetry.csv", parse_dates=["datetime"])
errors = pd.read_csv(RAW_DIR / "PdM_errors.csv", parse_dates=["datetime"])
failures = pd.read_csv(RAW_DIR / "PdM_failures.csv", parse_dates=["datetime"])
maint = pd.read_csv(RAW_DIR / "PdM_maint.csv", parse_dates=["datetime"])
machines = pd.read_csv(RAW_DIR / "PdM_machines.csv")
for _t in (tele, errors, failures, maint):                      # one datetime resolution everywhere (needed by merge_asof)
    _t["datetime"] = _t["datetime"].astype("datetime64[ns]")
tables = {"telemetry": tele, "errors": errors, "failures": failures, "maintenance": maint, "machines": machines}

# --- DQ-1: missing values and exact duplicate rows
audit = pd.DataFrame({n: {"rows": len(t), "missing": int(t.isna().sum().sum()), "duplicate_rows": int(t.duplicated().sum())}
                      for n, t in tables.items()}).T
print(audit.to_string())
record("DQ-1 no missing values and no duplicate rows", int(audit["missing"].sum() + audit["duplicate_rows"].sum()) == 0)

# --- DQ-2: telemetry is a complete, regular hourly grid for every machine
tele = tele.sort_values(["machineID", "datetime"]).reset_index(drop=True)
per_machine = tele.groupby("machineID")["datetime"].agg(["size", "min", "max"])
steps = tele.groupby("machineID")["datetime"].diff().dropna()
record("DQ-2 hourly grid, identical coverage for all machines",
       per_machine["size"].nunique() == 1 and (steps == pd.Timedelta(hours=1)).all()
       and not tele.duplicated(["machineID", "datetime"]).any(),
       f"{len(per_machine)} machines x {int(per_machine['size'].iloc[0]):,} rows, "
       f"{tele['datetime'].min()} -> {tele['datetime'].max()}")

# --- DQ-3: every failure timestamp lies on the telemetry grid of its machine
chk = failures.merge(tele[["machineID", "datetime"]], on=["machineID", "datetime"], how="left", indicator=True)
record("DQ-3 failure events align with telemetry timestamps", bool((chk["_merge"] == "both").all()),
       f"{len(failures)} failure rows")

# --- DQ-4: identifiers are consistent across tables
ids = set(machines["machineID"])
record("DQ-4 machine identifiers consistent across tables",
       set(tele["machineID"]) == ids and set(errors["machineID"]) <= ids
       and set(failures["machineID"]) <= ids and set(maint["machineID"]) <= ids,
       f"{len(ids)} machines")

# --- descriptive summary
fail_events = (failures[["machineID", "datetime"]].drop_duplicates()
               .sort_values(["machineID", "datetime"]).reset_index(drop=True))
print("\nDate ranges:")
for n, t in [("telemetry", tele), ("errors", errors), ("failures", failures), ("maintenance", maint)]:
    print(f"  {n:12s} {t['datetime'].min()} -> {t['datetime'].max()}")
print(f"\nFailure rows: {len(failures)} | distinct (machine, timestamp) failure events: {len(fail_events)} "
      f"| machines with failures: {failures['machineID'].nunique()}")
print("Failures by component:", failures["failure"].value_counts().sort_index().to_dict())
print("Machine models:", machines["model"].value_counts().sort_index().to_dict())

## 4. Target construction and verification

`label_failure_24h = 1` iff the next failure of the same machine occurs strictly after `t` and no later than `t + 24h`. A failure recorded exactly at `t` is **not** a future failure for row `t`.

The label is computed by one code path (`numpy.searchsorted`) and then **re-computed by two independent paths** (`merge_asof` and a brute-force check on a random sample). All three must agree exactly. The last 24 rows of each machine are removed because their full 24 h future is not observable (treating them as negatives would invent information).

In [ ]:
# 4.1  Label construction (primary path) + two independent verifications + boundary handling
def labels_searchsorted(tele_sorted, failures_df):
    """Primary path: first failure strictly after t via numpy.searchsorted."""
    lab = np.zeros(len(tele_sorted), dtype=np.int8)
    times = tele_sorted["datetime"].to_numpy()
    fails = {m: np.sort(g["datetime"].to_numpy()) for m, g in failures_df.groupby("machineID")}
    for m, idx in tele_sorted.groupby("machineID").indices.items():
        ft = fails.get(m)
        if ft is None:
            continue
        t = times[idx]
        pos = np.searchsorted(ft, t, side="right")          # index of first failure strictly after t
        ok = pos < len(ft)
        lab[idx[ok]] = ((ft[pos[ok]] - t[ok]) <= H64).astype(np.int8)
    return lab

def labels_merge_asof(tele_sorted, failures_df):
    """Independent path: pandas merge_asof (forward, exact matches excluded)."""
    left = tele_sorted[["machineID", "datetime"]].copy()
    left["_row"] = np.arange(len(left))
    left = left.sort_values("datetime")
    right = failures_df[["machineID", "datetime"]].drop_duplicates().rename(columns={"datetime": "fail_time"})
    right["datetime"] = right["fail_time"]
    right = right.sort_values("datetime")
    m = pd.merge_asof(left, right, on="datetime", by="machineID", direction="forward", allow_exact_matches=False)
    m = m.sort_values("_row")
    return ((m["fail_time"] - m["datetime"]) <= H).to_numpy().astype(np.int8)

def labels_bruteforce(tele_sorted, failures_df, n=3000, seed=SEED):
    """Independent path 2: direct definition on a random sample of rows."""
    rng = np.random.default_rng(seed)
    rows = rng.choice(len(tele_sorted), size=n, replace=False)
    fl = {m: g["datetime"].to_numpy() for m, g in failures_df.groupby("machineID")}
    out = []
    for r in rows:
        m, t = tele_sorted.at[r, "machineID"], tele_sorted.at[r, "datetime"].to_datetime64()
        ft = fl.get(m, np.array([], dtype="datetime64[ns]"))
        out.append(int(((ft > t) & (ft <= t + H64)).any()))
    return rows, np.array(out, dtype=np.int8)

lab_primary = labels_searchsorted(tele, failures)
lab_asof = labels_merge_asof(tele, failures)
rows_bf, lab_bf = labels_bruteforce(tele, failures)

record("L-1 label identical across three independent implementations",
       np.array_equal(lab_primary, lab_asof) and np.array_equal(lab_primary[rows_bf], lab_bf),
       f"searchsorted == merge_asof on {len(tele):,} rows; == brute force on {len(rows_bf):,} sampled rows")

tele[TARGET] = lab_primary

# Boundary handling: keep only rows whose complete 24 h future window lies inside the observation period
last_ts = tele.groupby("machineID")["datetime"].transform("max")
tele["observable"] = tele["datetime"] <= (last_ts - H)
n_dropped = int((~tele["observable"]).sum())
record("L-2 last 24 rows of every machine dropped (unobservable horizon)",
       (tele.groupby("machineID")["observable"].apply(lambda s: int((~s).sum())) == HORIZON_HOURS).all(),
       f"{n_dropped:,} rows dropped in total")

obs = tele[tele["observable"]]
pos, tot = int(obs[TARGET].sum()), len(obs)
print(f"\nCandidate rows: {tot:,} | positives: {pos:,} | negatives: {tot - pos:,} | positive rate: {100 * pos / tot:.4f}%")
print("Note: positives are NOT independent — one failure produces up to 24 consecutive positive rows.")

## 5. Simulated federated clients (factories)

The primary manifest has five clients with 20 machines each. Fixed per-model quotas create a deliberately non-IID machine-model distribution while ensuring that `factory_5` contains multiple models; this reduces, but does not eliminate, confounding between site and machine type. A second manifest preserves the stronger model-blocked shift scenario as a separate stress-test setup. Both are deterministic and versioned under `prep_v3/manifests/`.


In [ ]:
# 5.1  Deterministic mixed-model primary manifest + model-blocked stress manifest
FACTORY_ORDER = [f"factory_{i}" for i in range(1, N_FACTORIES + 1)]

# Fixed Azure dataset quotas. Column totals must match the dataset; row totals are 20.
# This creates non-IID model mixtures without making the primary held-out client a single-model client.
MODEL_QUOTAS = {
    "factory_1": {"model1": 8, "model2": 4, "model3": 4, "model4": 4},
    "factory_2": {"model1": 4, "model2": 7, "model3": 5, "model4": 4},
    "factory_3": {"model1": 2, "model2": 3, "model3": 10, "model4": 5},
    "factory_4": {"model1": 1, "model2": 2, "model3": 8, "model4": 9},
    "factory_5": {"model1": 1, "model2": 1, "model3": 8, "model4": 10},
}

observed_model_counts = machines["model"].value_counts().to_dict()
quota_model_counts = {
    model: sum(MODEL_QUOTAS[client].get(model, 0) for client in FACTORY_ORDER)
    for model in sorted({m for q in MODEL_QUOTAS.values() for m in q})
}
record("C-0 fixed partition quotas match the raw machine-model counts",
       observed_model_counts == quota_model_counts
       and all(sum(MODEL_QUOTAS[c].values()) == MACHINES_PER_FACTORY for c in FACTORY_ORDER),
       f"observed={observed_model_counts}; quota totals={quota_model_counts}")

assigned_parts = []
for j, model_name in enumerate(sorted(quota_model_counts)):
    model_rows = (machines[machines["model"] == model_name]
                  .sort_values(["age", "machineID"])
                  .sample(frac=1, random_state=SEED + j)
                  .reset_index(drop=True))
    cursor = 0
    for client in FACTORY_ORDER:
        n_take = MODEL_QUOTAS[client].get(model_name, 0)
        part = model_rows.iloc[cursor:cursor + n_take].copy()
        part["client_id"] = client
        assigned_parts.append(part)
        cursor += n_take
    assert cursor == len(model_rows), f"Quota allocation incomplete for {model_name}"

ordered = pd.concat(assigned_parts, ignore_index=True)
manifest = (ordered[["machineID", "client_id", "model", "age"]]
            .sort_values(["client_id", "machineID"]).reset_index(drop=True))
composition = pd.crosstab(manifest["client_id"], manifest["model"]).reindex(
    index=FACTORY_ORDER, columns=sorted(quota_model_counts), fill_value=0)
composition["mean_age"] = manifest.groupby("client_id")["age"].mean().reindex(FACTORY_ORDER).round(1)

record("C-1 five mixed-model factories x 20 machines; each machine assigned once",
       manifest["machineID"].is_unique
       and (manifest.groupby("client_id").size() == MACHINES_PER_FACTORY).all()
       and manifest["client_id"].nunique() == N_FACTORIES
       and (composition[sorted(quota_model_counts)].gt(0).sum(axis=1) >= 2).all()
       and (composition.loc[HELD_OUT, sorted(quota_model_counts)] > 0).sum() >= 3,
       "primary clients contain multiple models; factory_5 has at least three model types")

# Preserve an alternative severe machine-type shift: contiguous groups after sorting by model.
stress_order = machines.sort_values(["model", "age", "machineID"]).reset_index(drop=True).copy()
stress_order["client_id"] = [f"factory_{i // MACHINES_PER_FACTORY + 1}" for i in range(len(stress_order))]
stress_manifest = (stress_order[["machineID", "client_id", "model", "age"]]
                   .sort_values(["client_id", "machineID"]).reset_index(drop=True))

manifest.to_csv(MAN_DIR / "factory_manifest.csv", index=False)
manifest.to_json(MAN_DIR / "factory_manifest.json", orient="records", indent=2)
stress_manifest.to_csv(MAN_DIR / "factory_manifest_type_shift_stress.csv", index=False)

# Compare against the prior manifest for audit, but do not fail merely because the primary
# partition intentionally changed. The old file is never overwritten.
legacy_path = PROJECT_DIR / "manifests" / "factory_manifest.csv"
legacy_comparison = {"legacy_path": str(legacy_path), "legacy_found": legacy_path.exists(), "same_mapping": None}
if legacy_path.exists():
    legacy = pd.read_csv(legacy_path)[["machineID", "client_id"]]
    active_map = manifest[["machineID", "client_id"]]
    cmp_ = active_map.merge(legacy, on="machineID", suffixes=("", "_legacy"), validate="one_to_one")
    legacy_comparison["same_mapping"] = bool(len(cmp_) == 100 and (cmp_["client_id"] == cmp_["client_id_legacy"]).all())
    print("Prior manifest found:", legacy_path)
    print("Same machine-to-client mapping:", legacy_comparison["same_mapping"])
    print("The revised primary mapping is versioned in prep_v3; prior mapping remains untouched.")
else:
    print("No root-level legacy manifest found; prep_v3 manifest created.")
(MAN_DIR / "legacy_manifest_comparison.json").write_text(json.dumps(legacy_comparison, indent=2))
(CFG_DIR / "factory_partition_config.json").write_text(json.dumps({
    "primary_method": "fixed non-IID model quotas; deterministic shuffle within each model",
    "seed": SEED,
    "model_quotas": MODEL_QUOTAS,
    "observed_model_counts": observed_model_counts,
    "primary_manifest": "manifests/factory_manifest.csv",
    "alternate_stress_manifest": "manifests/factory_manifest_type_shift_stress.csv",
    "alternate_stress_note": "Model-blocked partition; not used in the primary splits in this notebook. Run as a separate experiment.",
}, indent=2))

print("\nPrimary machine-model composition per factory:")
print(composition.to_string())
print("\nStress-test model composition (model-blocked assignment):")
print(pd.crosstab(stress_manifest["client_id"], stress_manifest["model"]).to_string())
print("\nSaved primary manifest and alternate model-type-shift stress manifest under", MAN_DIR)


## 6. Splits: temporal windows, held-out factory, and calibration isolation

- **Development factories 1–4:** `train` → 24 h purge gap → `validation` → 24 h purge gap → `test_time_late`.
- **Held-out factory 5:** no supervised rows enter train/validation/tuning. Its first 50% of the timeline is a feature-only, unlabeled calibration window for the optional `local` scaler. This file excludes labels and is not an evaluation split.
- `test_site_full` contains all held-out-factory rows from the calibration cutoff onward, so no evaluation row is used to fit `local_factory_5`.
- `test_site_late` is the late-window subset of `test_site_full`; it is stricter, but the two tests are nested and are not statistically independent.
- `test_time_late` uses factories 1–4 in the same late window.

The primary factory assignment contains all machine types in every client, with intentionally different proportions. The separate model-blocked manifest is a stress-test alternative, not evidence of site-only generalization by itself.


In [ ]:
# 6.1  Build splits (metadata + labels only); held-out calibration keys are kept label-free
cand = (tele.loc[tele["observable"], ["datetime", "machineID", TARGET]]
        .merge(manifest[["machineID", "client_id"]], on="machineID", how="left", validate="many_to_one")
        .sort_values(["machineID", "datetime"]).reset_index(drop=True))
assert cand["client_id"].notna().all()

uts = np.sort(cand["datetime"].unique())
VAL_START = pd.Timestamp(uts[int(len(uts) * TRAIN_FRACTION)])
TRAIN_CUTOFF = VAL_START - H
remainder = uts[uts >= VAL_START.to_datetime64()]
TEST_START = pd.Timestamp(remainder[int(len(remainder) * VAL_FRACTION_OF_REMAINDER)])
VAL_CUTOFF = TEST_START - H
CALIBRATION_CUTOFF = pd.Timestamp(uts[int(len(uts) * HELD_OUT_CALIBRATION_FRACTION)])
assert CALIBRATION_CUTOFF + H < TRAIN_CUTOFF, "Calibration must end well before the development training cutoff"

dt = cand["datetime"]
is_dev = cand["client_id"].isin(TRAINING_FACTORIES)
is_ho = cand["client_id"].eq(HELD_OUT)
MASKS = {
    "train":          is_dev & (dt < TRAIN_CUTOFF),
    "validation":     is_dev & (dt >= VAL_START) & (dt < VAL_CUTOFF),
    "test_time_late": is_dev & (dt >= TEST_START),
    "test_site_late": is_ho & (dt >= TEST_START),
    "test_site_full": is_ho & (dt >= CALIBRATION_CUTOFF),
}
SPLIT_KEYS = {name: cand.loc[m].reset_index(drop=True) for name, m in MASKS.items()}
SPLIT_NAMES = list(SPLIT_KEYS)
# This calibration table intentionally omits TARGET to make label use impossible through this artifact.
CALIBRATION_KEYS = (cand.loc[is_ho & (dt < CALIBRATION_CUTOFF), ["datetime", "machineID", "client_id"]]
                    .reset_index(drop=True))

print("Time boundaries")
print(f"  train             : {SPLIT_KEYS['train']['datetime'].min()} -> {SPLIT_KEYS['train']['datetime'].max()}")
print(f"  validation        : {SPLIT_KEYS['validation']['datetime'].min()} -> {SPLIT_KEYS['validation']['datetime'].max()}")
print(f"  calibration cutoff: {CALIBRATION_CUTOFF} (fraction={HELD_OUT_CALIBRATION_FRACTION:.0%})")
print(f"  late test window  : {TEST_START} -> {cand['datetime'].max()}")

# --- temporal and site isolation verification
tr, va = SPLIT_KEYS["train"], SPLIT_KEYS["validation"]
record("S-1 24h purge gaps: no label window crosses train/validation/test boundary",
       tr["datetime"].max() + H < VAL_START and va["datetime"].max() + H < TEST_START,
       f"train end + 24h={tr['datetime'].max() + H} < val start={VAL_START}; "
       f"validation end + 24h={va['datetime'].max() + H} < test start={TEST_START}")

ho_machines = set(manifest.loc[manifest["client_id"] == HELD_OUT, "machineID"])
dev_machines = set(manifest.loc[manifest["client_id"].isin(TRAINING_FACTORIES), "machineID"])
record("S-2 held-out machines never appear in supervised development splits",
       ho_machines.isdisjoint(dev_machines)
       and all(HELD_OUT not in set(SPLIT_KEYS[s]["client_id"]) for s in ["train", "validation", "test_time_late"])
       and all(set(SPLIT_KEYS[s]["client_id"]) == {HELD_OUT} for s in ["test_site_late", "test_site_full"]),
       f"{len(ho_machines)} held-out machines isolated from train/validation/time-test rows")

record("S-3 no duplicate (machine, timestamp) rows inside any evaluation split",
       all(not d.duplicated(["machineID", "datetime"]).any() for d in SPLIT_KEYS.values())
       and cand.groupby("machineID")["client_id"].nunique().max() == 1)

late_in_full = SPLIT_KEYS["test_site_late"].merge(
    SPLIT_KEYS["test_site_full"], on=["machineID", "datetime"], how="left", indicator=True)
record("S-4 late site test is a subset of post-calibration site test; all evaluation splits contain positives",
       bool((late_in_full["_merge"] == "both").all())
       and all(d[TARGET].sum() > 0 for d in SPLIT_KEYS.values()),
       f"test_site_full rows={len(SPLIT_KEYS['test_site_full']):,}; late subset rows={len(SPLIT_KEYS['test_site_late']):,}")

calib_eval_overlap = CALIBRATION_KEYS[["machineID", "datetime"]].merge(
    SPLIT_KEYS["test_site_full"][["machineID", "datetime"]], on=["machineID", "datetime"], how="inner")
record("S-5 held-out calibration is earlier than and disjoint from all site-evaluation rows",
       len(CALIBRATION_KEYS) > 0 and calib_eval_overlap.empty
       and CALIBRATION_KEYS["datetime"].max() < SPLIT_KEYS["test_site_full"]["datetime"].min()
       and CALIBRATION_KEYS["datetime"].max() + H < TRAIN_CUTOFF,
       f"calibration rows={len(CALIBRATION_KEYS):,}; eval starts={SPLIT_KEYS['test_site_full']['datetime'].min()}")

# Informational continuity with the old version. Differences are expected because the primary
# factory assignment and site-test definition have been intentionally revised.
print("\nPrevious-v2 counts are not used as acceptance criteria; partition and held-out evaluation changed intentionally.")


### 6.2 How much evidence does each split contain?

Rows are not independent: one failure creates up to 24 positive rows. The real sample size is the number of **distinct failure events** covered by positive labels. Metrics (PR-AUC, recall, F1) computed on splits with few events are statistically fragile — report confidence intervals and prefer the splits with more events.

In [ ]:
# 6.2  Rows, positives and distinct failure events per split and per factory
def covered_events(df):
    """Distinct failure events (machine, timestamp) having >=1 row t in the split with t < f <= t + 24h."""
    ts = {m: np.sort(g["datetime"].to_numpy()) for m, g in df.groupby("machineID")}
    n = 0
    for m, g in fail_events.groupby("machineID"):
        if m not in ts:
            continue
        f = g["datetime"].to_numpy()
        lo = np.searchsorted(ts[m], f - H64, side="left")      # first row with t >= f - 24h
        hi = np.searchsorted(ts[m], f, side="left")            # first row with t >= f  (rows before are t < f)
        n += int(((hi - lo) > 0).sum())
    return n

rows = []
for split, df in SPLIT_KEYS.items():
    for client, g in list(df.groupby("client_id")) + [("ALL", df)]:
        ev = covered_events(g)
        rows.append({"split": split, "factory": client, "rows": len(g), "machines": g["machineID"].nunique(),
                     "positive_rows": int(g[TARGET].sum()), "positive_rate_pct": round(100 * g[TARGET].mean(), 3),
                     "failure_events": ev, "fragile(<%d events)" % FRAGILE_EVENT_THRESHOLD: ev < FRAGILE_EVENT_THRESHOLD})
EVENT_TABLE = pd.DataFrame(rows)
EVENT_TABLE.to_csv(REP_DIR / "split_event_table.csv", index=False)
print(EVENT_TABLE.to_string(index=False))
n_frag = int(EVENT_TABLE.iloc[:, -1].sum())
print(f"\n{n_frag} of {len(EVENT_TABLE)} split/factory cells have fewer than {FRAGILE_EVENT_THRESHOLD} distinct failure events — "
      "per-factory and late-window metrics there are statistically fragile; use confidence intervals.")
record("S-5 event table produced", len(EVENT_TABLE) > 0,
       f"{len(EVENT_TABLE)} rows saved to split_event_table.csv")

## 7. Maintenance records: time semantics (D2)

Maintenance records are only allowed as features if their timing is understood. This cell measures how often a maintenance (component replacement) record coincides with a failure of the same machine.

In [ ]:
# 7.1  Time semantics of maintenance records relative to failures
maint_k = maint.rename(columns={"comp": "component"})
fail_k = failures.rename(columns={"failure": "component"})

at_failure_any = maint_k.merge(fail_events, on=["machineID", "datetime"], how="inner")
at_failure_same = maint_k.merge(fail_k, on=["machineID", "datetime", "component"], how="inner")
fails_with_replacement = (fail_k.merge(maint_k, on=["machineID", "datetime", "component"], how="left", indicator=True)["_merge"]
                          .eq("both").mean())

MAINT_SEMANTICS = {
    "maintenance_records": len(maint),
    "records_before_telemetry_start": int((maint["datetime"] < tele["datetime"].min()).sum()),
    "records_at_a_failure_timestamp_any_component": len(at_failure_any),
    "records_at_a_failure_timestamp_same_component": len(at_failure_same),
    "share_of_failures_with_same_timestamp_replacement": round(float(fails_with_replacement), 4),
}
for k, v in MAINT_SEMANTICS.items():
    print(f"  {k:55s} {v}")

print("""
Interpretation
  * Maintenance records that coincide with a failure are reactive replacements. After such a failure,
    hours_since_last_maintenance resets, so the feature partly encodes 'time since the last failure'.
  * The feature uses only events STRICTLY BEFORE the prediction time, so rows that precede a failure
    never see that failure's replacement record: no label leakage.
  * Decision: keep maintenance recency in the primary feature set 'full' (information available in
    deployment) and ALSO save the feature set 'no_maintenance'; the later ablation study must report both.""")
record("D-2 maintenance time semantics measured and documented", True,
       f"{MAINT_SEMANTICS['share_of_failures_with_same_timestamp_replacement']:.1%} of failure rows have a same-timestamp replacement of the same component")

## 8. Past-only feature engineering

Five feature groups, every one computed from information available **at or before** the prediction time `t`:

| Group | Features |
|---|---|
| `sensor_raw` | `volt`, `rotate`, `pressure`, `vibration` at time `t` |
| `rolling_stats` | mean and std of each sensor over the previous 24 h and 7 d (the window **excludes** row `t`) |
| `error_recency` | `log1p(hours since the last error strictly before t)`, `has_prior_error` |
| `maintenance_recency` | `log1p(hours since the last maintenance record strictly before t)` |
| `history_counts` | number of past hourly observations available, capped at 24 and 168 |

Missing-history rules (only the sentinel is learned, and only from training rows): early rolling mean → the current reading; early rolling std → 0; "no prior error" → training-only sentinel (max observed training recency + 1 h) plus the `has_prior_error` indicator. Recency features are `log1p`-transformed to remove their heavy tail before scaling.

In [ ]:
# 8.1  Feature definitions (pure functions; reused by the leakage tests)
FEATURE_GROUPS = {
    "sensor_raw": list(SENSORS),
    "rolling_stats": [f"{s}_past_{stat}_{w}" for s in SENSORS for w in ROLLING_WINDOWS for stat in ("mean", "std")],
    "error_recency": ["hours_since_last_error_log1p", "has_prior_error"],
    "maintenance_recency": ["hours_since_last_maintenance_log1p"],
    "history_counts": [f"past_observation_count_{w}" for w in ROLLING_WINDOWS],
}
ALL_FEATURES = [c for g in FEATURE_GROUPS.values() for c in g]
NO_MAINT_FEATURES = [c for c in ALL_FEATURES if c not in FEATURE_GROUPS["maintenance_recency"]]
META_COLS = ["datetime", "machineID", "client_id"]
assert len(ALL_FEATURES) == 25 and len(set(ALL_FEATURES)) == 25

def _hours_since_last_event(keys, events):
    """Hours since the most recent event of the same machine STRICTLY BEFORE each row's timestamp (NaN if none)."""
    ev = events[["machineID", "datetime"]].rename(columns={"datetime": "event_time"})
    ev["datetime"] = ev["event_time"]
    ev = ev.sort_values("datetime")
    left = keys[["machineID", "datetime"]].copy()
    left["_row"] = np.arange(len(left))
    left = left.sort_values("datetime")
    m = pd.merge_asof(left, ev, on="datetime", by="machineID", direction="backward", allow_exact_matches=False)
    m = m.sort_values("_row")
    return ((m["datetime"] - m["event_time"]) / pd.Timedelta(hours=1)).to_numpy(dtype="float64")

def build_raw_features(tele_df, errors_df, maint_df):
    """Past-only features before any train-derived filling. Returns NaN where history is missing."""
    df = tele_df[["machineID", "datetime"] + SENSORS].sort_values(["machineID", "datetime"]).reset_index(drop=True)
    out = df.copy()
    for s in SENSORS:
        past = df.groupby("machineID", sort=False)[s].shift(1)              # strictly earlier reading
        grp = past.groupby(df["machineID"], sort=False)
        for wname, w in ROLLING_WINDOWS.items():
            out[f"{s}_past_mean_{wname}"] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
            out[f"{s}_past_std_{wname}"] = grp.rolling(w, min_periods=2).std(ddof=0).reset_index(level=0, drop=True)
    pos = df.groupby("machineID", sort=False).cumcount().to_numpy()
    for wname, w in ROLLING_WINDOWS.items():
        out[f"past_observation_count_{wname}"] = np.minimum(pos, w).astype("int16")
    out["hours_since_last_error_raw"] = _hours_since_last_event(df, errors_df)
    out["hours_since_last_maintenance_raw"] = _hours_since_last_event(df, maint_df)
    return out

def finalize_features(raw, error_sentinel_hours):
    """Apply the missing-history rules and log1p; returns keys + the 25 model features (float32)."""
    out = raw[["machineID", "datetime"]].copy()
    for s in SENSORS:
        out[s] = raw[s]
        for wname in ROLLING_WINDOWS:
            out[f"{s}_past_mean_{wname}"] = raw[f"{s}_past_mean_{wname}"].fillna(raw[s])      # current reading
            out[f"{s}_past_std_{wname}"] = raw[f"{s}_past_std_{wname}"].fillna(0.0)
    for wname in ROLLING_WINDOWS:
        out[f"past_observation_count_{wname}"] = raw[f"past_observation_count_{wname}"]
    e = raw["hours_since_last_error_raw"]
    out["has_prior_error"] = e.notna().astype("int8")
    out["hours_since_last_error_log1p"] = np.log1p(e.fillna(error_sentinel_hours))
    mt = raw["hours_since_last_maintenance_raw"]
    if mt.isna().any():
        raise ValueError("Unexpected rows without a prior maintenance record")
    out["hours_since_last_maintenance_log1p"] = np.log1p(mt)
    out[ALL_FEATURES] = out[ALL_FEATURES].astype("float32")
    return out[["machineID", "datetime"] + ALL_FEATURES]

print("Feature groups:", {g: len(c) for g, c in FEATURE_GROUPS.items()}, "| total:", len(ALL_FEATURES))

In [ ]:
# 8.2  Build features, derive the training-only sentinel, save evaluation splits and feature-only calibration data
raw_feat = build_raw_features(tele, errors, maint)
assert len(raw_feat) == len(tele)

# has_prior_maintenance would be constant (maintenance logs start before telemetry) -> excluded, verified here
record("FE-0 every telemetry row has a prior maintenance record (has_prior_maintenance is constant, excluded)",
       bool(raw_feat["hours_since_last_maintenance_raw"].notna().all()))

# Training-only sentinel for 'no prior error'
tr_err = SPLIT_KEYS["train"][["machineID", "datetime"]].merge(
    raw_feat[["machineID", "datetime", "hours_since_last_error_raw"]], on=["machineID", "datetime"], how="left", validate="one_to_one")
ERROR_SENTINEL_HOURS = float(tr_err["hours_since_last_error_raw"].dropna().max()) + 1.0
print(f"Training-only error-recency sentinel: {ERROR_SENTINEL_HOURS:.1f} h  (log1p -> {np.log1p(ERROR_SENTINEL_HOURS):.3f})")

feat_all = finalize_features(raw_feat, ERROR_SENTINEL_HOURS)

SPLIT_DF, fill_rows = {}, []
for name, keys in SPLIT_KEYS.items():
    df = keys.merge(feat_all, on=["machineID", "datetime"], how="left", validate="one_to_one")
    raw_part = keys[["machineID", "datetime"]].merge(raw_feat, on=["machineID", "datetime"], how="left", validate="one_to_one")
    mean_cols = [c for c in raw_part.columns if "_past_mean_" in c]; std_cols = [c for c in raw_part.columns if "_past_std_" in c]
    fill_rows.append({"split": name, "rows": len(df),
                      "rows_with_filled_rolling_mean": int(raw_part[mean_cols].isna().any(axis=1).sum()),
                      "rows_with_filled_rolling_std": int(raw_part[std_cols].isna().any(axis=1).sum()),
                      "rows_with_no_prior_error": int(raw_part["hours_since_last_error_raw"].isna().sum())})
    assert len(df) == len(keys), name
    assert np.isfinite(df[ALL_FEATURES].to_numpy()).all(), f"non-finite feature in {name}"
    df = df[META_COLS + [TARGET] + ALL_FEATURES]
    df.to_parquet(PROC_DIR / f"{name}.parquet", index=False)
    SPLIT_DF[name] = df

# Persist held-out calibration as feature-only data: no TARGET/label column is written.
calibration_df = CALIBRATION_KEYS.merge(
    feat_all, on=["machineID", "datetime"], how="left", validate="one_to_one")
calibration_df = calibration_df[META_COLS + ALL_FEATURES]
assert TARGET not in calibration_df.columns and np.isfinite(calibration_df[ALL_FEATURES].to_numpy()).all()
calibration_df.to_parquet(PROC_DIR / "heldout_calibration_features.parquet", index=False)

FILL_TABLE = pd.DataFrame(fill_rows)
print("\nRows affected by the missing-history rules:")
print(FILL_TABLE.to_string(index=False))
print(f"\nSaved feature-only held-out calibration rows: {len(calibration_df):,}; labels are absent.")

train_df = SPLIT_DF["train"]
constant = [c for c in ALL_FEATURES if train_df[c].nunique() <= 1]
record("FE-1 all 25 features finite, none constant in training", len(constant) == 0 and len(ALL_FEATURES) == 25,
       f"constant features: {constant or 'none'}")

FEATURE_CONFIG = {
    "groups": FEATURE_GROUPS,
    "feature_sets": {"full": ALL_FEATURES, "no_maintenance": NO_MAINT_FEATURES},
    "primary_feature_set": "full", "ablation_feature_set": "no_maintenance",
    "excluded_constant_features": ["has_prior_maintenance"],
    "identifier_columns": META_COLS, "target": TARGET,
    "error_recency_sentinel_hours": ERROR_SENTINEL_HOURS,
    "missing_history_rules": {"rolling_mean": "current sensor reading", "rolling_std": 0.0,
                              "no_prior_error": "training-only sentinel + has_prior_error indicator",
                              "recency": "log1p", "rolling_windows_rows": ROLLING_WINDOWS},
    "boundaries": {"train_cutoff": str(TRAIN_CUTOFF), "val_start": str(VAL_START), "val_cutoff": str(VAL_CUTOFF),
                   "test_start": str(TEST_START), "held_out_calibration_cutoff": str(CALIBRATION_CUTOFF),
                   "held_out_calibration_fraction": HELD_OUT_CALIBRATION_FRACTION,
                   "horizon_hours": HORIZON_HOURS},
}
(CFG_DIR / "feature_groups.json").write_text(json.dumps(FEATURE_CONFIG, indent=2))
FILL_TABLE.to_csv(REP_DIR / "missing_history_fill_counts.csv", index=False)
print("\nSaved:", ", ".join(f"{n}.parquet" for n in SPLIT_DF), "+ heldout_calibration_features.parquet + configs/feature_groups.json")

## 9. Normalization policies

Both policies are fitted on the allowed rows and saved. The held-out local scaler uses only the earlier, feature-only calibration file; the calibration file has no target column and is disjoint from `test_site_full` and `test_site_late`.

| Policy | Scalers | Held-out factory |
|---|---|---|
| `global` | One scaler fitted on pooled supervised training rows from factories 1–4 | Same global scaler; no feature rows from factory 5 used for fitting |
| `local` | One scaler per development factory fitted on that factory's training rows | `local_factory_5` fitted on `heldout_calibration_features.parquet`, containing only features and metadata before `CALIBRATION_CUTOFF` |

This local policy represents **unsupervised site adaptation**, not zero-shot normalization. `test_site_full` begins at the calibration cutoff, so evaluation rows do not participate in fitting. `test_site_late` is nested within `test_site_full` and must not be treated as an independent sample.


In [ ]:
# 9.1  Fit and save scalers on their explicitly allowed source rows
SPLIT_DF = {n: pd.read_parquet(PROC_DIR / f"{n}.parquet") for n in SPLIT_NAMES}
train_df = SPLIT_DF["train"]
calib = pd.read_parquet(PROC_DIR / "heldout_calibration_features.parquet")

record("N-0 calibration artifact contains no target and no evaluation rows",
       TARGET not in calib.columns
       and list(calib.columns) == META_COLS + ALL_FEATURES
       and not calib.duplicated(["machineID", "datetime"]).any()
       and calib["datetime"].max() < SPLIT_DF["test_site_full"]["datetime"].min(),
       f"{len(calib):,} feature-only calibration rows; evaluation starts {SPLIT_DF['test_site_full']['datetime'].min()}")

def fit_scaler(df):
    sc = StandardScaler().fit(df[ALL_FEATURES].to_numpy(dtype="float64"))
    assert (sc.var_ > 0).all(), "zero-variance feature"
    return sc

SCALERS, SCALER_SOURCES = {}, {}
SCALERS["global"] = fit_scaler(train_df)
SCALER_SOURCES["global"] = f"train rows of {TRAINING_FACTORIES}"
for f in TRAINING_FACTORIES:
    SCALERS[f"local_{f}"] = fit_scaler(train_df[train_df["client_id"] == f])
    SCALER_SOURCES[f"local_{f}"] = f"train rows of {f} only"

SCALERS[f"local_{HELD_OUT}"] = fit_scaler(calib)
SCALER_SOURCES[f"local_{HELD_OUT}"] = (
    f"feature-only unlabeled {HELD_OUT} calibration rows with datetime < {CALIBRATION_CUTOFF}")

stats = {}
for name, sc in SCALERS.items():
    joblib.dump(sc, SCALER_DIR / f"{name}.joblib")
    stats[name] = {"fit_source": SCALER_SOURCES[name], "n_rows": int(sc.n_samples_seen_),
                   "features": ALL_FEATURES, "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist()}
(CFG_DIR / "scaler_statistics.json").write_text(json.dumps(stats, indent=2))

PROTOCOL = {
    "method": "StandardScaler (per column), fitted in float64 on saved float32 feature values",
    "policies": {
        "global": "one scaler on pooled training rows of factories 1-4; applied to every evaluation split",
        "local": "one scaler per client; held-out factory uses feature-only unlabeled rows before CALIBRATION_CUTOFF",
    },
    "held_out_calibration": {
        "file": "processed/heldout_calibration_features.parquet",
        "cutoff": str(CALIBRATION_CUTOFF),
        "fraction_of_timeline": HELD_OUT_CALIBRATION_FRACTION,
        "contains_target": False,
        "evaluation_disjoint": True,
    },
    "never_fitted_on": ["validation rows for supervised clients", "evaluation rows", "any evaluation labels"],
    "note_test_site_full": "Contains only held-out rows at/after CALIBRATION_CUTOFF; local scaler calibration rows are strictly earlier and saved without labels.",
    "feature_set_selection": "select columns after scaling (feature_sets in feature_groups.json)",
}
(CFG_DIR / "preprocessing_protocol.json").write_text(json.dumps(PROTOCOL, indent=2))

summary = pd.DataFrame([{"scaler": n, "rows": s["n_rows"], "fit_source": s["fit_source"][:90]} for n, s in stats.items()])
print(summary.to_string(index=False))
record("N-1 six scalers fitted from allowed sources and saved", len(SCALERS) == 1 + N_FACTORIES,
       "global + 4 local training factories + local held-out feature-only calibration")


## 10. Loader for the next phase

This cell writes `prep_v3/fedkg_data.py`, a small loader used by every later notebook, so that all experiments read the data and apply the normalization policy in exactly the same way, then tests it.

```python
import sys; sys.path.insert(0, "/content/drive/MyDrive/FedKG-Maint/prep_v3")
from fedkg_data import load_split
X, y, meta = load_split("train", policy="global", feature_set="full")
```

In [ ]:
# 10.1  Write and test the loader module
MODULE_SRC = '''"""Loader for the prepared FedKG-Maint data (v3).

    import sys; sys.path.insert(0, "<PROJECT_DIR>/prep_v3")
    from fedkg_data import load_split
    X, y, meta = load_split("train", policy="global", feature_set="full")
"""
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent
SPLITS = ("train", "validation", "test_time_late", "test_site_late", "test_site_full")
META_COLS = ["datetime", "machineID", "client_id"]
TARGET = "label_failure_24h"


def load_split(split, policy="global", feature_set="full", root=ROOT):
    """Return (X float32, y float32, meta DataFrame).

    policy:      "global" | "local" | "none"  (normalization policy, see configs/preprocessing_protocol.json)
    feature_set: "full" (25 features) | "no_maintenance" (24 features)
    """
    root = Path(root)
    cfg = json.loads((root / "configs" / "feature_groups.json").read_text())
    all_cols = cfg["feature_sets"]["full"]
    keep = [all_cols.index(c) for c in cfg["feature_sets"][feature_set]]
    df = pd.read_parquet(root / "processed" / f"{split}.parquet")
    X = df[all_cols].to_numpy(dtype="float64")
    if policy == "global":
        X = joblib.load(root / "scalers" / "global.joblib").transform(X)
    elif policy == "local":
        scaled = np.empty_like(X)
        for client in df["client_id"].unique():
            m = (df["client_id"] == client).to_numpy()
            scaled[m] = joblib.load(root / "scalers" / f"local_{client}.joblib").transform(X[m])
        X = scaled
    elif policy != "none":
        raise ValueError("policy must be 'global', 'local' or 'none'")
    return X[:, keep].astype("float32"), df[TARGET].to_numpy("float32"), df[META_COLS].reset_index(drop=True)


def load_client(split, client_id, **kwargs):
    """Rows of one factory only (what a federated client would hold)."""
    X, y, meta = load_split(split, **kwargs)
    m = (meta["client_id"] == client_id).to_numpy()
    return X[m], y[m], meta[m].reset_index(drop=True)
'''
(PREP_DIR / "fedkg_data.py").write_text(MODULE_SRC)

import importlib
sys.path.insert(0, str(PREP_DIR))
import fedkg_data
importlib.reload(fedkg_data)

report = []
for split in fedkg_data.SPLITS:
    for policy in ("global", "local", "none"):
        for fs, n_feat in (("full", 25), ("no_maintenance", 24)):
            X, y, meta = fedkg_data.load_split(split, policy=policy, feature_set=fs)
            assert X.shape == (len(y), n_feat) and np.isfinite(X).all() and len(meta) == len(y)
    report.append((split, X.shape[0]))

# the scaled training data must have mean 0 / std 1 (global) and per factory (local)
Xg, _, _ = fedkg_data.load_split("train", "global")
Xl, _, ml = fedkg_data.load_split("train", "local")
ok_global = np.allclose(Xg.mean(0), 0, atol=1e-3) and np.allclose(Xg.std(0), 1, atol=1e-3)
ok_local = all(np.allclose(Xl[(ml["client_id"] == f).to_numpy()].mean(0), 0, atol=1e-3) for f in TRAINING_FACTORIES)
Xc, yc, mc = fedkg_data.load_client("train", "factory_1", policy="local")
record("N-2 loader works for every split x policy x feature set; scaling behaves as specified",
       ok_global and ok_local and set(mc["client_id"]) == {"factory_1"}, f"{dict(report)}")

## 11. Leakage and integrity tests

- **F1 — future-perturbation:** for sampled rows from every evaluation split, changing sensor readings after `t` and moving error/maintenance events at or after `t` must leave the feature vector at `t` unchanged. A positive control perturbs the past and must change the features.
- **F2 — scaler source:** JSON and `.joblib` scaler statistics are recomputed from their permitted source rows. The held-out scaler source is the feature-only calibration artifact, not an evaluation split.
- **F3 — evaluation separation:** the held-out calibration artifact contains no target column and has no machine/timestamp keys in common with `test_site_full`.
- **F4 — schema:** all evaluation splits have identical columns, no identifiers or target appear among model features, and the feature groups match the saved config.


In [ ]:
# 11.1  Leakage and integrity tests
rng = np.random.default_rng(SEED)
N_PER_SPLIT, N_CONTROL = 200, 10

tele_m = {m: g[["machineID", "datetime"] + SENSORS].reset_index(drop=True) for m, g in tele.groupby("machineID")}
err_m = {m: g[["machineID", "datetime"]].reset_index(drop=True) for m, g in errors.groupby("machineID")}
mnt_m = {m: g[["machineID", "datetime"]].reset_index(drop=True) for m, g in maint.groupby("machineID")}
EMPTY = pd.DataFrame({"machineID": pd.Series(dtype="int64"), "datetime": pd.Series(dtype="datetime64[ns]")})

def recompute_row(split, i, mode):
    """Recompute the 25 features of row i after perturbing the future ('future') or the past ('past')."""
    row = SPLIT_DF[split].iloc[i]
    m, t = int(row["machineID"]), row["datetime"]
    tl, er, mt = tele_m[m].copy(), err_m.get(m, EMPTY).copy(), mnt_m.get(m, EMPTY).copy()
    if mode == "future":
        f = (tl["datetime"] > t).to_numpy()
        tl.loc[f, SENSORS] = rng.normal(1000, 500, size=(int(f.sum()), len(SENSORS)))
        for ev in (er, mt):
            g = (ev["datetime"] >= t).to_numpy()
            new_times = (t + pd.to_timedelta(rng.integers(0, 3000, int(g.sum())), unit="h")).astype("datetime64[ns]")
            ev.loc[g, "datetime"] = new_times.to_numpy()
        extra = pd.DataFrame({"machineID": [m, m], "datetime": pd.to_datetime([t, t + pd.Timedelta(hours=1)]).astype("datetime64[ns]")})
        er, mt = pd.concat([er, extra], ignore_index=True), pd.concat([mt, extra], ignore_index=True)
    else:  # past perturbation (positive control)
        p = ((tl["datetime"] < t) & (tl["datetime"] >= t - pd.Timedelta(hours=48))).to_numpy()
        tl.loc[p, SENSORS] = rng.normal(1000, 500, size=(int(p.sum()), len(SENSORS)))
    fin = finalize_features(build_raw_features(tl, er, mt), ERROR_SENTINEL_HOURS)
    new = fin.loc[fin["datetime"] == t, ALL_FEATURES].to_numpy(dtype="float64")[0]
    return new, row[ALL_FEATURES].to_numpy(dtype="float64")

n_tests = n_bad = 0
for split in SPLIT_NAMES:
    for i in rng.choice(len(SPLIT_DF[split]), size=min(N_PER_SPLIT, len(SPLIT_DF[split])), replace=False):
        new, stored = recompute_row(split, int(i), "future")
        n_tests += 1
        n_bad += int(not np.allclose(new, stored, rtol=1e-5, atol=1e-5))
record("F-1 future perturbation leaves every tested feature vector unchanged", n_bad == 0,
       f"{n_tests} rows tested ({N_PER_SPLIT} per split), {n_bad} changed")

# positive control: perturbing the PAST must change the features (rows with a full 24 h history)
changed = tested = 0
tr_ok = np.flatnonzero(SPLIT_DF["train"]["past_observation_count_24h"].to_numpy() >= 24)
for i in rng.choice(tr_ok, size=N_CONTROL, replace=False):
    new, stored = recompute_row("train", int(i), "past")
    tested += 1
    changed += int(not np.allclose(new, stored, rtol=1e-5, atol=1e-5))
record("F-2 positive control: perturbing the past does change the features", changed == tested, f"{changed}/{tested} rows changed")

# --- F2/F3: scaler statistics are verified against the exact allowed source rows;
# held-out calibration is feature-only and disjoint from site evaluation.
saved = json.loads((CFG_DIR / "scaler_statistics.json").read_text())
disk = {n: pd.read_parquet(PROC_DIR / f"{n}.parquet") for n in ("train", "test_site_full")}
calibration_disk = pd.read_parquet(PROC_DIR / "heldout_calibration_features.parquet")

def source_rows(name):
    if name == "global":
        return disk["train"]
    if name == f"local_{HELD_OUT}":
        return calibration_disk
    return disk["train"][disk["train"]["client_id"] == name.replace("local_", "")]

calib_eval_keys = calibration_disk[["machineID", "datetime"]].merge(
    disk["test_site_full"][["machineID", "datetime"]], on=["machineID", "datetime"], how="inner")
calibration_valid = (
    TARGET not in calibration_disk.columns
    and list(calibration_disk.columns) == META_COLS + ALL_FEATURES
    and len(calib_eval_keys) == 0
    and calibration_disk["datetime"].max() < disk["test_site_full"]["datetime"].min()
)
ok = bool(calibration_valid)
for name, st in saved.items():
    X = source_rows(name)[ALL_FEATURES].to_numpy(dtype="float64")
    obj = joblib.load(SCALER_DIR / f"{name}.joblib")
    ok &= np.allclose(X.mean(0), st["mean"], rtol=1e-9, atol=1e-9)
    ok &= np.allclose(X.std(0), st["scale"], rtol=1e-9, atol=1e-9)
    ok &= np.allclose(obj.mean_, st["mean"]) and np.allclose(obj.scale_, st["scale"])
record("F-3 scaler stats match permitted source rows; held-out calibration is label-free and disjoint", bool(ok),
       f"{len(saved)} scalers checked; calibration rows={len(calibration_disk):,}; overlap rows={len(calib_eval_keys)}")

# --- F3: schema and feature/identifier separation
cols_ref = list(pd.read_parquet(PROC_DIR / "train.parquet").columns)
same_cols = all(list(pd.read_parquet(PROC_DIR / f"{n}.parquet").columns) == cols_ref for n in SPLIT_NAMES)
cfg = json.loads((CFG_DIR / "feature_groups.json").read_text())
record("F-4 identical schema in every split; features match feature_groups.json; no id/target leaks into features",
       same_cols and cfg["feature_sets"]["full"] == ALL_FEATURES
       and set(ALL_FEATURES).isdisjoint(META_COLS + [TARGET, "has_prior_maintenance"])
       and cols_ref == META_COLS + [TARGET] + ALL_FEATURES,
       f"{len(ALL_FEATURES)} features; no_maintenance set has {len(cfg['feature_sets']['no_maintenance'])}")

## 12. Final report

Writes `reports/data_prep_final_report.json` and `.md` with the PASS/FAIL checklist, split sizes, event counts, manifest details, and known limitations. A failed check stops the run. Because the split/partition protocol changed in v3, the previous v2 execution outputs are intentionally not carried over; run this notebook from top to bottom in the intended environment.


In [ ]:
# 12.1  Final report
import datetime as _dt

LIMITATIONS = [
    "The factories are simulated clients, not real factory records; this preprocessing notebook centralizes the raw tables to simulate the partition.",
    "The primary partition uses mixed machine types with different quotas, reducing the old single-model held-out confounding but not proving generalization to a real site. A model-blocked stress-test manifest is saved separately and must be evaluated in a separate experiment.",
    "The first 50% of the held-out timeline is used only for unsupervised local-scaler calibration. The feature-only artifact omits labels, and all site-evaluation rows are later than the calibration cutoff.",
    "test_site_late is a subset of test_site_full; these results are nested, not independent samples. Confidence intervals should be computed by machine or distinct failure event, not by treating hourly rows as independent.",
    "The data cover one year and each temporal split has a limited number of distinct failure events; per-factory and late-window metrics can be statistically fragile.",
    "Positive rows are not independent: one failure can produce up to 24 consecutive positive rows. The effective sample size is closer to the number of distinct failure events.",
    "Maintenance recency partly encodes time since the last failure because replacements may coincide with failures. It is past-only, included in 'full', and should be ablated using 'no_maintenance'.",
    "Rolling windows are row-count based and rely on the verified regular hourly grid (DQ-2).",
    "A global scaler is a centralized reference. The local scaler is an unsupervised adaptation policy; neither alone implements secure federated learning or a formal privacy guarantee.",
]

PROBLEM_TABLE = [
    ("Raw-data reproducibility", "Compare SHA-256, byte size and row count with an immutable prior manifest; fail on mismatch", "DATA-1, DATA-2"),
    ("Held-out calibration overlaps old site test", "Use a feature-only calibration artifact and start site evaluation after the calibration cutoff", "S-5, N-0, F-3"),
    ("Site/type confounding", "Use mixed-model quotas for primary clients and save a separate model-blocked stress-test manifest", "C-0, C-1, factory_partition_config.json"),
    ("Label construction", "Recompute the 24-hour label independently and test boundary handling", "L-1, L-2"),
    ("Temporal leakage", "Use 24-hour purge gaps, past-only features, future perturbation and a past-perturbation positive control", "S-1, F-1, F-2"),
    ("Feature/schema integrity", "Validate finite values, scaler source statistics, consistent schemas and exclusion of labels/IDs from features", "FE-1, F-3, F-4"),
    ("Machine-type stress generalization", "Save a model-blocked alternate manifest; it must be run as a separate evaluation scenario", "factory_manifest_type_shift_stress.csv"),
]

report = {
    "generated_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"),
    "notebook_version": "prep_v3_revised",
    "environment": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
    "checks": CHECKS,
    "all_checks_pass": all(c["status"] == "PASS" for c in CHECKS.values()),
    "boundaries": FEATURE_CONFIG["boundaries"],
    "factory_partition": {
        "primary_manifest": "manifests/factory_manifest.csv",
        "alternate_stress_manifest": "manifests/factory_manifest_type_shift_stress.csv",
        "model_quotas": MODEL_QUOTAS,
        "composition": composition.reset_index().to_dict(orient="records"),
        "legacy_comparison": legacy_comparison,
    },
    "normalization": {
        "policies": ["global", "local", "none"],
        "held_out_calibration_cutoff": str(CALIBRATION_CUTOFF),
        "held_out_calibration_fraction": HELD_OUT_CALIBRATION_FRACTION,
        "calibration_file": "processed/heldout_calibration_features.parquet",
        "calibration_contains_target": False,
        "evaluation_disjoint_from_calibration": True,
    },
    "splits": {n: {"rows": len(d), "machines": int(d["machineID"].nunique()),
                   "positive_rows": int(d[TARGET].sum()),
                   "positive_rate_pct": round(100 * d[TARGET].mean(), 4),
                   "factories": sorted(d["client_id"].unique())}
               for n, d in SPLIT_DF.items()},
    "held_out_calibration": {"rows": int(len(calibration_df)), "columns": list(calibration_df.columns),
                              "contains_target": TARGET in calibration_df.columns},
    "failure_events": EVENT_TABLE[EVENT_TABLE["factory"] == "ALL"].set_index("split")["failure_events"].to_dict(),
    "features": {"n_features": len(ALL_FEATURES), "groups": {g: len(c) for g, c in FEATURE_GROUPS.items()},
                 "primary_feature_set": "full", "ablation_feature_set": "no_maintenance",
                 "error_recency_sentinel_hours": ERROR_SENTINEL_HOURS},
    "maintenance_semantics": MAINT_SEMANTICS,
    "raw_data_checksums": checksums,
    "known_limitations": LIMITATIONS,
    "problem_to_change_to_evidence": [{"problem": a, "change": b, "evidence": c} for a, b, c in PROBLEM_TABLE],
}
(REP_DIR / "data_prep_final_report.json").write_text(json.dumps(report, indent=2, default=str))

md = ["# FedKG-Maint — Data preparation v3 final report", "", f"Generated (UTC): {report['generated_utc']}", "",
      "## Checklist", "", "| Check | Status | Detail |", "|---|---|---|"]
md += [f"| {k} | {v['status']} | {v['detail'][:140]} |" for k, v in CHECKS.items()]
md += ["", "## Final evaluation splits", "", "| Split | Rows | Machines | Positive rows | Positive rate % | Failure events |", "|---|---:|---:|---:|---:|---:|"]
md += [f"| {n} | {s['rows']:,} | {s['machines']} | {s['positive_rows']:,} | {s['positive_rate_pct']} | {report['failure_events'][n]} |"
       for n, s in report["splits"].items()]
md += ["", "## Held-out calibration", "",
       f"- Calibration cutoff: `{CALIBRATION_CUTOFF}`",
       f"- Rows: {len(calibration_df):,}",
       f"- Target column present: `{TARGET in calibration_df.columns}`",
       "- The held-out local scaler uses this feature-only file; evaluation rows begin at/after the cutoff.",
       "", "## Problem → change → evidence", "", "| Problem | Change | Evidence |", "|---|---|---|"]
md += [f"| {a} | {b} | {c} |" for a, b, c in PROBLEM_TABLE]
md += ["", "## Known limitations", ""] + [f"{i}. {l}" for i, l in enumerate(LIMITATIONS, 1)]
(REP_DIR / "data_prep_final_report.md").write_text("\n".join(md))

print(pd.DataFrame([{"check": k, "status": v["status"]} for k, v in CHECKS.items()]).to_string(index=False))
print("\nFinal evaluation splits:")
print(pd.DataFrame(report["splits"]).T[["rows", "machines", "positive_rows", "positive_rate_pct"]].assign(
    failure_events=pd.Series(report["failure_events"])).to_string())
print("\nHeld-out calibration:", len(calibration_df), "rows; target column absent:", TARGET not in calibration_df.columns)
print("\nReport saved to", REP_DIR / "data_prep_final_report.md")
assert report["all_checks_pass"], "At least one check failed"
print("\nDATA PREPARATION v3 COMPLETE — all checks PASS.")


## Done — what comes next

Point the next notebook to `prep_v3/` and retrain its baselines: the primary factory assignment and `test_site_full` definition have changed, so Phase 2 results from `prep_v2/` are not directly comparable.

```python
import sys
sys.path.insert(0, "/content/drive/MyDrive/FedKG-Maint/prep_v3")
from fedkg_data import load_split, load_client

X_tr, y_tr, m_tr = load_split("train", policy="global", feature_set="full")
X_va, y_va, m_va = load_split("validation", policy="global", feature_set="full")
X_ho, y_ho, m_ho = load_split("test_site_full", policy="global", feature_set="full")
```

Rules for Phase 2: tune only on `validation`; evaluate the three test sets after model selection; compare both feature sets and normalization policies; report PR-AUC, thresholded precision/recall/F1, event counts and confidence intervals. The model-blocked stress-test manifest is an alternate experiment and must not be conflated with the primary result.
